# Qwen3-32B 퀴즈 생성 Fine-tuning

## 목표
- 스터디 요약 내용 → JSON 형식 퀴즈 생성
- 객관식, OX, 단답형 혼합 출제

## 메모리 최적화
- **MAX_SEQ_LENGTH**: 1024 (퀴즈 출력이 길어서 요약보다 큼)
- **LoRA**: r=8
- **Batch**: 1 x 16

## 환경
- **GPU**: SSAFY L40S 46GB
- **모델**: Qwen3-32B (4-bit)
- **최적화**: Unsloth + LoRA + bf16

In [ ]:
# ===== 0. 환경 설정 =====
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import torch
import json
import numpy as np
from sklearn.model_selection import train_test_split

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

---
## 1. 데이터 로드 및 분할 (70/15/15)

In [ ]:
# 퀴즈 학습 데이터 로드
with open("quiz_training_data.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"전체 데이터: {len(raw_data)}개")

# 샘플 확인
sample = raw_data[0]
print(f"\n[샘플 구조]")
for msg in sample["messages"]:
    print(f"  {msg['role']}: {msg['content'][:100]}...")

# ===== Train/Val/Test 분할 =====
train_data, temp_data = train_test_split(raw_data, test_size=0.3, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

print(f"\n[데이터 분할]")
print(f"  Train:      {len(train_data):>5}개 (70%)")
print(f"  Validation: {len(val_data):>5}개 (15%)")
print(f"  Test:       {len(test_data):>5}개 (15%)")

# Test 데이터 저장
with open("quiz_test_data.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, ensure_ascii=False, indent=2)
print(f"\nTest 데이터 저장: quiz_test_data.json")

---
## 2. 모델 로드

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen3-32B-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 1024  # 퀴즈 출력이 길어서 요약(512)보다 큼

print(f"모델 로딩: {MODEL_NAME}")
print("4-bit 양자화로 로딩 중...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

print(f"\n모델 로드 완료!")
print(f"VRAM 사용: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

---
## 3. [BEFORE] 파인튜닝 전 성능 테스트

In [ ]:
# 퀴즈 생성 테스트 샘플 (고정 - Before/After 비교용)
TEST_SUMMARIES = [
    """Docker 기초 스터디 요약:
- Docker 컨테이너와 이미지의 차이점 학습
- Dockerfile 작성법 (FROM, RUN, COPY, CMD)
- docker-compose로 멀티 컨테이너 관리
- 볼륨 마운트와 네트워크 설정""",

    """알고리즘 스터디 요약:
- 다이나믹 프로그래밍 기본 개념 (메모이제이션 vs 타뷸레이션)
- 피보나치 수열로 DP 이해
- 배낭 문제(Knapsack) 풀이
- 최장 공통 부분 수열(LCS) 알고리즘"""
]

QUIZ_PROMPT_TEMPLATE = """<|im_start|>system
당신은 IT 스터디 내용을 바탕으로 복습 퀴즈를 생성하는 전문가입니다.
반드시 JSON 배열 형식으로 출력하세요.<|im_end|>
<|im_start|>user
다음 스터디 요약 내용을 바탕으로 복습 퀴즈 3문제를 JSON 형식으로 생성해주세요.

스터디 요약:
{summary}<|im_end|>
<|im_start|>assistant
"""

def generate_quiz(model, tokenizer, summary, max_new_tokens=512):
    prompt = QUIZ_PROMPT_TEMPLATE.format(summary=summary)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # assistant 응답 부분만 추출
    if "<|im_start|>assistant" in response:
        response = response.split("<|im_start|>assistant")[-1]
    return response.strip()

In [ ]:
# Before 결과 저장
FastLanguageModel.for_inference(model)

print("=" * 60)
print("[BEFORE] 파인튜닝 전 퀴즈 생성")
print("=" * 60)

before_results = []
for i, summary in enumerate(TEST_SUMMARIES):
    print(f"\n--- 샘플 {i+1} ---")
    result = generate_quiz(model, tokenizer, summary)
    before_results.append(result)
    print(result[:500] if len(result) > 500 else result)

---
## 4. LoRA 설정

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=8,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("LoRA 설정 완료 (r=8, 32B 극한 모드)")
model.print_trainable_parameters()

---
## 5. 데이터셋 준비 (ChatML 포맷)

In [ ]:
from datasets import Dataset

def format_chatml(item):
    """ChatML 포맷으로 변환"""
    messages = item["messages"]
    text = ""
    for msg in messages:
        role = msg["role"]
        content = msg["content"]
        text += f"<|im_start|>{role}\n{content}<|im_end|>\n"
    return {"text": text}

# Train/Val 데이터셋 생성
train_dataset = Dataset.from_list([format_chatml(item) for item in train_data])
val_dataset = Dataset.from_list([format_chatml(item) for item in val_data])

# 토큰 길이 확인
sample_tokens = tokenizer(train_dataset[0]['text'])
print(f"Train 데이터셋: {len(train_dataset)}개")
print(f"Val 데이터셋: {len(val_dataset)}개")
print(f"샘플 토큰 수: {len(sample_tokens['input_ids'])}")

# 토큰 길이 분포 확인
lengths = [len(tokenizer(d['text'])['input_ids']) for d in train_dataset.select(range(min(100, len(train_dataset))))]
print(f"\n[토큰 길이 분포 (100개 샘플)]")
print(f"  평균: {np.mean(lengths):.0f}")
print(f"  최대: {max(lengths)}")
print(f"  최소: {min(lengths)}")
print(f"  1024 초과: {sum(1 for l in lengths if l > 1024)}개")

print(f"\n[샘플]\n{train_dataset[0]['text'][:300]}...")

---
## 6. Trainer 설정

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen3-quiz-lora",
    
    # === 32B 극한 설정 ===
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    
    # === 최적화 ===
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    fp16=False,
    bf16=True,
    
    # === 평가/저장 ===
    eval_strategy="no",
    save_strategy="no",
    
    # === 로깅 ===
    logging_steps=5,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

print("[Trainer 설정 - 32B 극한 모드]")
print(f"  - MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")
print(f"  - Batch: 1 x 16")
print(f"  - Epochs: 1")

---
## 7. 학습 실행

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

print("=" * 60)
print("퀴즈 모델 학습 시작")
print("=" * 60)
print(f"VRAM: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

train_result = trainer.train()

print("\n" + "=" * 60)
print("학습 완료!")
print("=" * 60)
print(f"Total steps: {train_result.global_step}")
print(f"Final train loss: {train_result.training_loss:.4f}")

---
## 8. 학습 곡선 시각화

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train_loss = [(x['step'], x['loss']) for x in logs if 'loss' in x]

plt.figure(figsize=(10, 5))

if train_loss:
    steps, losses = zip(*train_loss)
    plt.plot(steps, losses, label='Train Loss', marker='o', alpha=0.7)
    
    min_idx = np.argmin(losses)
    plt.scatter([steps[min_idx]], [losses[min_idx]], color='red', s=100, zorder=5)
    plt.annotate(f'Min: {losses[min_idx]:.4f}', 
                 xy=(steps[min_idx], losses[min_idx]),
                 xytext=(10, 10), textcoords='offset points')

plt.xlabel('Steps')
plt.ylabel('Loss')
plt.title('Quiz Model - Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('quiz_training_curve.png', dpi=150)
plt.show()

if train_loss:
    print(f"Final Loss: {losses[-1]:.4f}")
    print(f"Min Loss: {min(losses):.4f} (step {steps[np.argmin(losses)]})")

---
## 9. [AFTER] 파인튜닝 후 성능 테스트

In [ ]:
FastLanguageModel.for_inference(model)

print("=" * 60)
print("[AFTER] 파인튜닝 후 퀴즈 생성")
print("=" * 60)

after_results = []
for i, summary in enumerate(TEST_SUMMARIES):
    print(f"\n--- 샘플 {i+1} ---")
    result = generate_quiz(model, tokenizer, summary)
    after_results.append(result)
    print(result[:500] if len(result) > 500 else result)

---
## 10. Before/After 비교 + JSON 유효성 검사

In [ ]:
def check_json_valid(text):
    """JSON 파싱 가능 여부 체크"""
    try:
        # markdown 코드블록 제거
        clean = text
        if "```json" in clean:
            clean = clean.split("```json")[1].split("```")[0].strip()
        elif "```" in clean:
            clean = clean.split("```")[1].split("```")[0].strip()
        
        parsed = json.loads(clean)
        if isinstance(parsed, list):
            return True, len(parsed), parsed
        return True, 1, parsed
    except:
        return False, 0, None

print("=" * 70)
print("[COMPARISON] Before vs After")
print("=" * 70)

for i in range(len(TEST_SUMMARIES)):
    print(f"\n{'='*70}")
    print(f"샘플 {i+1}")
    print(f"{'='*70}")
    print(f"\n[INPUT]")
    print(TEST_SUMMARIES[i][:150] + "...")
    
    # Before
    b_valid, b_count, _ = check_json_valid(before_results[i])
    print(f"\n[BEFORE] JSON 유효: {b_valid}, 문제 수: {b_count}")
    print(before_results[i][:300] + "..." if len(before_results[i]) > 300 else before_results[i])
    
    # After
    a_valid, a_count, _ = check_json_valid(after_results[i])
    print(f"\n[AFTER] JSON 유효: {a_valid}, 문제 수: {a_count}")
    print(after_results[i][:300] + "..." if len(after_results[i]) > 300 else after_results[i])

---
## 11. Test 데이터 정량 평가

In [ ]:
# Test 데이터에서 JSON 유효성 + 문제 수 정확도 측정
print("=" * 60)
print("[Test 데이터 정량 평가]")
print("=" * 60)

eval_count = min(50, len(test_data))  # 최대 50개 평가
json_valid = 0
total_questions = 0

for i in range(eval_count):
    item = test_data[i]
    # user 메시지에서 요약 추출
    user_msg = next((m["content"] for m in item["messages"] if m["role"] == "user"), "")
    
    result = generate_quiz(model, tokenizer, user_msg, max_new_tokens=512)
    valid, count, _ = check_json_valid(result)
    
    if valid:
        json_valid += 1
        total_questions += count
    
    if (i + 1) % 10 == 0:
        print(f"  진행: {i+1}/{eval_count} (JSON 유효: {json_valid}/{i+1})")

print(f"\n[결과]")
print(f"  JSON 유효율: {json_valid}/{eval_count} ({json_valid/eval_count:.1%})")
print(f"  평균 문제 수: {total_questions/max(json_valid,1):.1f}개")

---
## 12. 모델 저장

In [ ]:
# LoRA 어댑터 저장
SAVE_PATH = "./qwen3-quiz-lora"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"모델 저장 완료: {SAVE_PATH}")
print(f"\n[저장된 파일]")
for f in os.listdir(SAVE_PATH):
    size = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1024 / 1024
    print(f"  {f}: {size:.1f} MB")

---
## 13. GGUF 변환 (Merged Model)

In [ ]:
# LoRA를 base 모델에 merge 후 GGUF로 변환
print("GGUF 변환 시작...")
print("LoRA merge + Q4_K_M 양자화")

model.save_pretrained_gguf(
    "./qwen3-quiz-gguf",
    tokenizer,
    quantization_method="q4_k_m",
)

print("\nGGUF 변환 완료!")
gguf_dir = "./qwen3-quiz-gguf"
for f in os.listdir(gguf_dir):
    if f.endswith(".gguf"):
        size = os.path.getsize(os.path.join(gguf_dir, f)) / 1024 / 1024 / 1024
        print(f"  {f}: {size:.1f} GB")

---
## 14. 최종 리포트

In [ ]:
print("\n" + "=" * 60)
print("퀴즈 모델 학습 리포트")
print("=" * 60)

print(f"\n[모델]")
print(f"  Base: {MODEL_NAME}")
print(f"  Method: LoRA (r=8, alpha=8)")
print(f"  Quantization: 4-bit")
print(f"  Max Seq Length: {MAX_SEQ_LENGTH}")

print(f"\n[데이터]")
print(f"  Train: {len(train_data)}개 (70%)")
print(f"  Val:   {len(val_data)}개 (15%)")
print(f"  Test:  {len(test_data)}개 (15%)")

print(f"\n[학습]")
print(f"  Epochs: 1")
print(f"  Steps: {train_result.global_step}")
print(f"  Train Loss: {train_result.training_loss:.4f}")
print(f"  Batch: 1 x 16")

print(f"\n[저장]")
print(f"  LoRA: {SAVE_PATH}")
print(f"  GGUF: ./qwen3-quiz-gguf/")

print("\n" + "=" * 60)
print("완료! GGUF 파일을 추론 서버에 배포하세요.")
print("  scp qwen3-quiz-gguf/*.gguf 서버:/models/")
print("=" * 60)